# Ella2027 — LTX-2.5 22B on Kaggle T4

A clean, reproducible Kaggle notebook for **LTX-2.5 22B Distilled NVFP4 + Gemma 4 12B INT8 ConvRot**, using Wan2GP. Large checkpoints are stored under `/tmp/ella-ltx25`, never in `/kaggle/working`.

**First test:** 480p, about 5 seconds, batch size 1. Audio/video generation is supported by the model; confirm audio is enabled in the Wan2GP generation settings.

> Kaggle T4 does not provide native optimized NVFP4 kernels. Wan2GP may use a slower fallback. This notebook does not promise a particular generation time or output quality.

## 1. Preflight — GPU, internet, and storage
Run this first. It does not download model weights.

In [ ]:
import os, shutil, torch, subprocess, sys
print('Python:', sys.version)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU accelerator before continuing.')
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'GPU {i}: {p.name} | VRAM {p.total_memory/1024**3:.2f} GB')
MODEL_DIR = '/tmp/ella-ltx25'
os.makedirs(MODEL_DIR, exist_ok=True)
for path in ['/tmp', '/kaggle/working']:
    u = shutil.disk_usage(path)
    print(f'{path}: free={u.free/1024**3:.1f} GB | total={u.total/1024**3:.1f} GB')
print('Model directory:', MODEL_DIR)
print('Preflight complete.')

## 2. Get Wan2GP source
The repository is cloned into the small working area. Model weights are stored separately in `/tmp`.

In [ ]:
import os, subprocess
WGP_DIR = '/kaggle/working/Wan2GP'
if not os.path.isdir(os.path.join(WGP_DIR, '.git')):
    if os.path.exists(WGP_DIR):
        raise RuntimeError(f'{WGP_DIR} exists but is not a Git checkout. Rename/remove it before continuing.')
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/deepbeepmeep/Wan2GP.git', WGP_DIR], check=True)
else:
    print('Wan2GP checkout already exists; keeping it unchanged.')
os.chdir(WGP_DIR)
subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=True)
print('Wan2GP source ready:', WGP_DIR)

## 3. Install dependencies without replacing Kaggle's CUDA PyTorch
Pin the already-installed Torch family during requirements installation. This avoids an unnecessary large Torch/CUDA replacement.

In [ ]:
import os, sys, subprocess, torch
import importlib.util
from pathlib import Path
os.chdir(WGP_DIR)
constraints = Path('/tmp/ella-kaggle-constraints.txt')
pins = [f'torch=={torch.__version__.split("+")[0]}']
for package in ['torchvision', 'torchaudio']:
    if importlib.util.find_spec(package) is not None:
        mod = __import__(package)
        pins.append(f'{package}=={mod.__version__.split("+")[0]}')
constraints.write_text('\n'.join(pins) + '\n')
print('Constraints:', constraints.read_text())
if not Path('requirements.txt').is_file():
    raise FileNotFoundError('Wan2GP requirements.txt was not found.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '-c', str(constraints), '-r', 'requirements.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', 'huggingface_hub>=0.30.0'], check=True)
print('Dependency installation complete. If Kaggle asks for a runtime restart, restart once and rerun cells from section 1.')

## 4. Download the two required LTX-2.5 checkpoints
Downloads are resumable through Hugging Face Hub's cache. If a download is interrupted, rerun this cell; completed files are reused. Do not save these large files in `/kaggle/working`.

In [ ]:
import os
from pathlib import Path
from huggingface_hub import hf_hub_download
MODEL_DIR = '/tmp/ella-ltx25'
os.makedirs(MODEL_DIR, exist_ok=True)
main_name = 'ltx-2.5-22b-distilled_diffusion_model_nvfp4.safetensors'
encoder_name = 'gemma4-12b-ltx-v1_int8_convrot.safetensors'
print('Downloading main LTX-2.5 checkpoint (large file; resumable)...')
main_path = hf_hub_download(repo_id='DeepBeepMeep/LTX-2', filename=main_name, local_dir=MODEL_DIR)
print('Main checkpoint ready:', main_path)
print('Downloading Gemma 4 12B INT8 ConvRot text encoder...')
encoder_path = hf_hub_download(repo_id='DeepBeepMeep/LTX-2', filename='gemma4-12b-ltx-v1/' + encoder_name, local_dir=MODEL_DIR)
print('Text encoder ready:', encoder_path)
for p in [Path(main_path), Path(encoder_path)]:
    if not p.is_file() or p.stat().st_size < 100_000_000:
        raise RuntimeError(f'Checkpoint missing or unexpectedly small: {p}')
    print(f'VERIFIED {p.name}: {p.stat().st_size/1024**3:.2f} GB')
print('\nBoth required checkpoints are present.')

## 5. Launch Wan2GP
Keep this cell running while generating. Open the public Gradio URL printed below.

In [ ]:
import os, subprocess, sys
os.chdir(WGP_DIR)
print('LTX-2.5 main checkpoint:', '/tmp/ella-ltx25/ltx-2.5-22b-distilled_diffusion_model_nvfp4.safetensors')
print('Gemma text encoder:', '/tmp/ella-ltx25/gemma4-12b-ltx-v1/gemma4-12b-ltx-v1_int8_convrot.safetensors')
print('Launch Wan2GP; keep this cell running.')
subprocess.run([sys.executable, 'wgp.py', '--share'], check=True)

## 6. First generation settings inside Wan2GP

1. Select **LTX-2.5 22B Distilled NVFP4**.
2. Open its model/checkpoint configuration.
3. Set the main checkpoint to `/tmp/ella-ltx25/ltx-2.5-22b-distilled_diffusion_model_nvfp4.safetensors`.
4. Set the text encoder to `/tmp/ella-ltx25/gemma4-12b-ltx-v1/gemma4-12b-ltx-v1_int8_convrot.safetensors`.
5. Make sure the local checkpoint paths are selected so Wan2GP does not download duplicates.
6. Start with 480p, around 5 seconds, batch size 1, and the model's distilled/default step setting.
7. Enable audio generation / audio output in the workflow settings, then verify the resulting MP4 contains an audio stream.

**Important:** Wan2GP may automatically fetch additional LTX-2.5 support files (VAEs, vocoder, connectors, and upscalers) when a workflow needs them. Allow those downloads to complete.

## Storage and recovery notes

- `/tmp/ella-ltx25` is temporary and is cleared when the Kaggle runtime is replaced. Keep a persistent Kaggle Dataset copy if you want to avoid downloading the large checkpoints again in future sessions.
- `/kaggle/working` is reserved for the small Wan2GP source and outputs; do not put the model weights there.
- If a model download is interrupted, rerun section 4. Hugging Face Hub reuses completed cached files and resumes partial transfers where supported.
- This notebook launches Wan2GP's Gradio UI. It is **not yet an Ella2027 HTTP worker** and does not connect to the app queue. Integration comes only after a successful audio/video test.
- T4 NVFP4 execution uses a fallback without optimized native NVFP4 kernels and can be slow. No specific generation time is guaranteed.